# Fetch Ingested Videos
Fetches recently (last 24 hours) ingested videos and prints 10 random videos with views.

In [ ]:
# Install pymongo if not already present
!pip install pymongo dnspython

import pymongo
from pymongo import MongoClient
from google.colab import userdata

try:
    # Retrieve the URI from Colab Secrets
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)

    # Send a ping to confirm a successful connection
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(f"An error occurred: {e}")
    print("\nMake sure you have added 'MONGODB_URI' to your Colab Secrets (the key icon on the left).")

In [ ]:
from bson.objectid import ObjectId
import datetime
import random

# Predikto database and collection mapping per documentation (assumed generic Colab use case since this is an analytics notebook similar to Graphiko)
db = client['predikto']
videos_col = db['publicvideos']

# Fetch videos ingested in the last 24 hours using timezone-aware UTC datetime
twenty_four_hours_ago = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=1)
oid = ObjectId.from_datetime(twenty_four_hours_ago)
query = {'_id': {'$gte': oid}}

# Use aggregate to randomly sample 10 videos
pipeline = [
    {'$match': query},
    {'$sample': {'size': 10}}
]
recent_videos = list(videos_col.aggregate(pipeline))

if not recent_videos:
    print('No videos found in the last 24 hours.')
else:
    print(f'Found {len(recent_videos)} videos in the random sample. Printing titles and view counts:')
    for video in recent_videos:
        title = video.get('title', 'Unknown Title')
        views = video.get('viewCount', 0)
        print(f'- {title} | Views: {views}')
